In [ ]:
import pandas as pd
import random
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import os
from PIL import Image
import copy
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Dataset


from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

import torch.optim as optim

import cv2 
import copy
import time
%matplotlib inline

In [ ]:
def set_seed(seed=42):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    print(f"[INFO] Seed: {seed}")

set_seed(42)

[INFO] Seed: 42


# Load data

In [3]:
df = pd.read_csv('/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_metadata.csv')
df

,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear
...,...,...,...,...,...,...,...
10010,HAM_0002867,ISIC_0033084,akiec,histo,40.0,male,abdomen
10011,HAM_0002867,ISIC_0033550,akiec,histo,40.0,male,abdomen
10012,HAM_0002867,ISIC_0033536,akiec,histo,40.0,male,abdomen
10013,HAM_0000239,ISIC_0032854,akiec,histo,80.0,male,face


In [4]:
le = LabelEncoder()
dx_encoded = le.fit_transform(df['dx'])
df['dx_encoded'] = dx_encoded

# EDA

**Only images will be used for classification.**

In [5]:
df['dx_encoded'].value_counts()

dx_encoded
5    6705
4    1113
2    1099
1     514
0     327
6     142
3     115
Name: count, dtype: int64

**We have a class imbalance**

Plan
1. Resnet(18/152),   
   DenseNet(121/161),   
   ConvNeXt(tiny/large),   
   RegNet(y_400mf/x_32gf),   
   MobileNetV3(small/large),   
   ShuffleNetV2(x0_5/x2_0),   
   EfficientNet(b0/b7),   
   EfficientNetV2(s/l),   
   VisionTransformer(b_16/h_14),   
   SwinTransformer(t/v2_b),  
   DeiT/ConvNeXtV2/CaiT (after) with gradual unfreeze\learning rate decay without oversampling\weighted loss
2. add weighted loss
3. add oversampling
4. Compare training\inference time, accuracy
5. add table feature

ResNeXt101_64, 

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    df[['image_id',	'age',	'sex',	'localization']], 
    df['dx_encoded'],
    test_size=0.2,
    random_state=42,
    stratify=df['dx']
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, 
    y_train,
    test_size=0.1,
    random_state=42,
    stratify=y_train
)

In [7]:
part_1 = '/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1'
part_2 = '/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_2'

part_1_files = [os.path.join(part_1, f) for f in os.listdir(part_1) if os.path.getsize(os.path.join(part_1, f)) > 0 and f.endswith('.jpg')]
part_2_files = [os.path.join(part_2, f) for f in os.listdir(part_2) if os.path.getsize(os.path.join(part_2, f)) > 0 and f.endswith('.jpg')]

all_files = part_1_files + part_2_files

In [8]:
df_all_files_img = pd.DataFrame({'dir_img':all_files})
df_all_files_img['image_id'] = df_all_files_img['dir_img'].apply(lambda x: x.split('/')[5].split('.')[0])
df_all_files_img

,dir_img,image_id
0,/kaggle/input/skin-cancer-mnist-ham10000/HAM10...,ISIC_0028933
1,/kaggle/input/skin-cancer-mnist-ham10000/HAM10...,ISIC_0028394
2,/kaggle/input/skin-cancer-mnist-ham10000/HAM10...,ISIC_0027799
3,/kaggle/input/skin-cancer-mnist-ham10000/HAM10...,ISIC_0028100
4,/kaggle/input/skin-cancer-mnist-ham10000/HAM10...,ISIC_0027960
...,...,...
10010,/kaggle/input/skin-cancer-mnist-ham10000/HAM10...,ISIC_0029733
10011,/kaggle/input/skin-cancer-mnist-ham10000/HAM10...,ISIC_0033470
10012,/kaggle/input/skin-cancer-mnist-ham10000/HAM10...,ISIC_0032153
10013,/kaggle/input/skin-cancer-mnist-ham10000/HAM10...,ISIC_0030216


In [9]:
X_train_img = list(X_train.merge(df_all_files_img,
                            how='inner',
                            on='image_id')['dir_img'])

X_val_img = list(X_val.merge(df_all_files_img,
                            how='inner',
                            on='image_id')['dir_img'])

X_test_img = list(X_test.merge(df_all_files_img,
                            how='inner',
                            on='image_id')['dir_img'])

# Function for auto training

In [10]:
#Create dataset
class SkinDataset(Dataset):
    def __init__(self, filepaths, labels, transform=None):
        self.filepaths = filepaths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img_path = self.filepaths[idx]
        image = Image.open(img_path).convert('RGB')
        
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

In [ ]:
MODEL_CONFIG = {
    "resnet18":        (models.resnet18,        models.ResNet18_Weights.IMAGENET1K_V1, 224),
    "resnet152":       (models.resnet152,       models.ResNet152_Weights.IMAGENET1K_V2, 224),
    "densenet121":     (models.densenet121,     models.DenseNet121_Weights.IMAGENET1K_V1, 224),
    "densenet161":     (models.densenet161,     models.DenseNet161_Weights.IMAGENET1K_V1, 224),
    "convnext_tiny":   (models.convnext_tiny,   models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1, 224),
    "convnext_large":  (models.convnext_large,  models.ConvNeXt_Large_Weights.IMAGENET1K_V1, 384),
    "regnet_y_400mf":  (models.regnet_y_400mf,  models.RegNet_Y_400MF_Weights.IMAGENET1K_V2, 224),
    "regnet_y_128gf":  (models.regnet_y_128gf,  models.RegNet_Y_128GF_Weights.IMAGENET1K_SWAG_E2E_V1, 384),
    "mobilenet_v3_large": (models.mobilenet_v3_large, models.MobileNet_V3_Large_Weights.IMAGENET1K_V2, 224),
    "mobilenet_v3_small": (models.mobilenet_v3_small, models.MobileNet_V3_Small_Weights.IMAGENET1K_V1, 224),
    "shufflenet_v2_x0_5": (models.shufflenet_v2_x0_5, models.ShuffleNet_V2_X0_5_Weights.IMAGENET1K_V1, 224),
    "shufflenet_v2_x2_0": (models.shufflenet_v2_x2_0, models.ShuffleNet_V2_X2_0_Weights.IMAGENET1K_V1, 224),
    "efficientnet_b0": (models.efficientnet_b0, models.EfficientNet_B0_Weights.IMAGENET1K_V1, 224),
    "efficientnet_b7": (models.efficientnet_b7, models.EfficientNet_B7_Weights.IMAGENET1K_V1, 600),
    "efficientnetv2_s": (models.efficientnet_v2_s, models.EfficientNet_V2_S_Weights.IMAGENET1K_V1, 384),
    "efficientnetv2_l": (models.efficientnet_v2_l, models.EfficientNet_V2_L_Weights.IMAGENET1K_V1, 480),
    "vit_b_16":        (models.vit_b_16,        models.ViT_B_16_Weights.IMAGENET1K_V1, 224),
    "vit_h_14":        (models.vit_h_14,        models.ViT_H_14_Weights.IMAGENET1K_SWAG_E2E_V1, 518),
    "swin_t":          (models.swin_t,          models.Swin_T_Weights.IMAGENET1K_V1, 224),
    "swin_b":          (models.swin_b,          models.Swin_B_Weights.IMAGENET1K_V1, 384)
}

In [ ]:
def build_model_and_transforms(
    model_name: str,
    num_classes: int,
    pretrained: bool = True,
    strong_aug: bool = True,
    device: str = "cpu"
):
    """
    Универсальный билд для модели и трансформаций:
    - Автоматически подбирает официальные transforms
    - Создаёт классификатор под num_classes
    - Работает для всех архитектур torchvision 0.21.0+
    """
    model_name = model_name.lower()

    # === 1. Загружаем веса ===
    try:
        weights_enum = models.get_model_weights(model_name)
        weights = weights_enum.DEFAULT if pretrained else None
        print(f"[INFO] Loaded weights for {model_name}: {weights_enum.__name__}")
    except Exception as e:
        print(f"[WARN] Can't load weights for {model_name}: {e}")
        weights = None

    # === 2. Создаём модель ===
    model = models.get_model(model_name, weights=weights)

    # === 3. Меняем последний слой под num_classes ===
    if hasattr(model, "fc"):  # ResNet, DenseNet
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)

    elif hasattr(model, "classifier"):  # EfficientNet, ConvNeXt, MobileNet
        if isinstance(model.classifier, nn.Sequential):
            in_features = model.classifier[-1].in_features
            model.classifier[-1] = nn.Linear(in_features, num_classes)
        else:
            in_features = model.classifier.in_features
            model.classifier = nn.Linear(in_features, num_classes)

    elif hasattr(model, "head"):  # ViT, Swin
        in_features = model.head.in_features
        model.head = nn.Linear(in_features, num_classes)

    else:
        raise ValueError(f"❌ Неизвестная архитектура: {model_name}")

    # === 4. Загружаем официальные transforms ===
    if weights is not None:
        base_tfms = weights.transforms()
        mean, std = base_tfms.mean, base_tfms.std
        interpolation = base_tfms.interpolation
        crop_size = base_tfms.crop_size[0]
        resize_size = base_tfms.resize_size[0]
    else:
        # fallback
        mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
        interpolation = T.InterpolationMode.BICUBIC
        crop_size, resize_size = 224, 236

    # === 5. Аугментации ===
    train_tfms = [
        T.RandomResizedCrop(crop_size, scale=(0.8, 1.0),
                            ratio=(0.75, 1.33),
                            interpolation=interpolation),
        T.RandomHorizontalFlip()
    ]
    if strong_aug:
        train_tfms.append(
            T.RandomApply([T.ColorJitter(0.3, 0.3, 0.3, 0.1)], p=0.8)
        )

    train_tfms.extend([
        T.ToTensor(),
        T.Normalize(mean, std),
        T.RandomErasing(p=0.25)
    ])
    train_transform = T.Compose(train_tfms)

    val_test_transform = (
        weights.transforms() if weights is not None else
        T.Compose([
            T.Resize(resize_size, interpolation=interpolation),
            T.CenterCrop(crop_size),
            T.ToTensor(),
            T.Normalize(mean, std)
        ])
    )

    model = model.to(device)
    return model, train_transform, val_test_transform

In [ ]:
def freeze_backbone_unfreeze_head(model):
    # All freeze
    for p in model.parameters():
        p.requires_grad = False

    # unfreeze head / fc / classifier
    if hasattr(model, "fc"):
        for p in model.fc.parameters():
            p.requires_grad = True
    elif hasattr(model, "classifier"):
        for p in model.classifier.parameters():
            p.requires_grad = True
    elif hasattr(model, "head"):
        for p in model.head.parameters():
            p.requires_grad = True

    return model

In [ ]:
def gradual_unfreeze(model, model_type=None, epoch=0, optimizer=None, every=5, lr=1e-4):
    """
    Универсальный Gradual Unfreeze для популярных архитектур torchvision.
    Размораживает backbone по слоям каждые `every` эпох, начиная с верхних блоков.
    Лог выводится только когда действительно размораживается новый блок.
    """
    def add_to_optimizer(params):
        """Добавить новые параметры в оптимизатор, если их ещё нет."""
        if optimizer is not None:
            existing = {p for g in optimizer.param_groups for p in g["params"]}
            new_params = [p for p in params if p not in existing]
            if new_params:
                optimizer.add_param_group({"params": new_params, "lr": lr})

    # === Определяем архитектурные группы ===
    groups = []
    arch = model_type.lower() if model_type else ""

    # --- ResNet ---
    if hasattr(model, "layer4"):
        groups = [model.layer1, model.layer2, model.layer3, model.layer4]

    # --- EfficientNet / MobileNet / ConvNeXt / ShuffleNet ---
    elif hasattr(model, "features"):
        groups = list(model.features.children())
        if any(x in arch for x in ["efficientnet", "mobilenet", "shufflenet", "convnext"]):
            groups = list(reversed(groups))

    # --- DenseNet ---
    elif "densenet" in arch:
        groups = list(model.features.children())

    # --- RegNet ---
    elif "regnet" in arch and hasattr(model, "trunk_output"):
        blocks = list(model.trunk_output.children())
        groups = [b for b in blocks if isinstance(b, torch.nn.Sequential)]

    # --- Vision Transformer ---
    elif hasattr(model, "encoder") and hasattr(model.encoder, "layers"):
        groups = list(model.encoder.layers)

    # --- Swin Transformer ---
    elif hasattr(model, "stages"):
        groups = list(model.stages)

    if not groups:
        print(f"[WARN] Gradual unfreeze: не удалось определить блоки для {model_type}.")
        return

    # === Логика разморозки ===
    step = epoch // every
    if step == 0:
        return  # первые `every` эпох ничего не размораживаем

    n_to_unfreeze = min(step, len(groups))

    # Проверим, не были ли эти блоки уже разморожены
    already_unfrozen = sum(
        all(p.requires_grad for p in block.parameters()) for block in groups
    )

    if n_to_unfreeze <= already_unfrozen:
        return  # нечего размораживать, лог не печатаем

    # === Размораживаем только новые блоки ===
    new_blocks = groups[-n_to_unfreeze : -already_unfrozen or None]

    for block in new_blocks:
        for p in block.parameters():
            if not p.requires_grad:
                p.requires_grad = True
        add_to_optimizer(block.parameters())

    print(f"[INFO] Gradual unfreeze ({model_type}): разморожено {n_to_unfreeze}/{len(groups)} блоков.")


In [ ]:
def train_model(
    model,
    train_loader,
    val_loader,
    num_epochs,
    criterion,
    optimizer,
    patience=5,
    unfreeze_fn=None,
    model_type=None,
    device="cpu",
    scheduler=None,
    name_for_save="best_model",
    grad_clip=1.0,
):
    """
    Универсальная функция обучения модели.
    Поддерживает gradual unfreeze, early stopping и ReduceLROnPlateau.
    """

    train_losses, train_accuracies = [], []
    val_losses, val_accuracies = [], []

    best_val_accuracy = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())
    iter_without_improvements = 0

    model.to(device)

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch + 1}/{num_epochs}")

        # === Gradual unfreeze (если задан) ===
        if unfreeze_fn is not None:
            unfreeze_fn(model, model_type, epoch, optimizer)

        # === TRAIN ===
        model.train()
        train_loss, correct, total = 0.0, 0, 0

        for images, labels in tqdm(train_loader, desc=f"Train [{epoch+1}]"):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad(set_to_none=True)
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()

            # --- NEW: gradient clipping для стабильности ---
            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            optimizer.step()

            train_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

        train_loss /= total
        train_accuracy = correct / total
        train_losses.append(train_loss)
        train_accuracies.append(train_accuracy)

        # === VALIDATION ===
        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc=f"Val [{epoch+1}]"):
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                _, predicted = outputs.max(1)
                correct += predicted.eq(labels).sum().item()
                total += labels.size(0)

        val_loss /= total
        val_accuracy = correct / total
        val_losses.append(val_loss)
        val_accuracies.append(val_accuracy)

        print(f"Train: loss={train_loss:.4f}, acc={train_accuracy:.4f} | "
              f"Val: loss={val_loss:.4f}, acc={val_accuracy:.4f}")

        # === Scheduler ===
        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss)
            else:
                scheduler.step()

        # === Early stopping ===
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model_wts = copy.deepcopy(model.state_dict())
            iter_without_improvements = 0
            torch.save(model.state_dict(), f"{name_for_save}_state.pth")
        else:
            iter_without_improvements += 1
            if iter_without_improvements > patience:
                print(f"⏹ Early stopping at epoch {epoch + 1}")
                break

    # === Итог ===
    model.load_state_dict(best_model_wts)
    torch.save(model, f"{name_for_save}.pth")

    print(f"✅ Best val acc: {best_val_accuracy:.4f}")
    return model, train_losses, train_accuracies, val_losses, val_accuracies

# Training

In [ ]:
def get_training_config(architecture: str):
    """
    Возвращает оптимальные гиперпараметры под архитектуру:
    - batch_size (ориентир на 12–16 ГБ GPU или CPU)
    - learning rate
    - num_epochs
    - lr_decay_patience
    """
    arch = architecture.lower()

    # === Лёгкие архитектуры: быстро обучаются даже на CPU ===
    if any(x in arch for x in [
        "resnet18",
        "mobilenet_v3_small",
        "shufflenet",
        "regnet_y_400mf"
    ]):
        return dict(batch_size=64, lr=1e-3, num_epochs=20, patience=3)

    # === Средние архитектуры: умеренные требования к GPU ===
    elif any(x in arch for x in [
        "resnet152",
        "densenet121", "densenet161",
        "convnext_tiny",
        "efficientnet_b0",
        "efficientnetv2_s",
        "swin_t",
        "regnet_y_128gf",
        "mobilenet_v3_large"
    ]):
        return dict(batch_size=32, lr=1e-3, num_epochs=25, patience=3)

    # === Тяжёлые архитектуры: ViT, большие EfficientNet, Swin-B ===
    elif any(x in arch for x in [
        "efficientnet_b7",
        "efficientnetv2_m",
        "vit_b_16",
        "swin_b"
    ]):
        return dict(batch_size=16, lr=5e-4, num_epochs=25, patience=4)

    # === Очень тяжёлые архитектуры: ViT-H, ConvNeXt-L, EffNetV2-L ===
    elif any(x in arch for x in [
        "efficientnetv2_l",
        "vit_h_14",
        "convnext_large"
    ]):
        return dict(batch_size=8, lr=1e-4, num_epochs=30, patience=5)

    # === Запасной вариант для новых моделей ===
    else:
        return dict(batch_size=32, lr=1e-3, num_epochs=25, patience=3)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_classes = 7
criterion = nn.CrossEntropyLoss()

models_acr = [
    #"resnet18",           # my pc
    #"resnet152",          # my pc/kaggle
    #"densenet121",        # my pc/kaggle
    #"densenet161",        # my pc/kaggle
    #"convnext_tiny",      # my pc/kaggle
    #"convnext_large",     # rent cloud
    "regnet_y_400mf",     # my pc
    "regnet_y_128gf",     # my pc/kaggle
    #"mobilenet_v3_large", # my pc
    #"mobilenet_v3_small", # my pc
    #"shufflenet_v2_x0_5", # my pc
    #"shufflenet_v2_x2_0", # my pc
    "efficientnet_b0",    # kaggle
    "efficientnet_b7",    # rent cloud
    "efficientnetv2_s",   # kaggle
    "efficientnetv2_l",   # rent cloud
    "vit_b_16",           # kaggle/colab
    "vit_h_14",           # rent cloud
    "swin_t",             # kaggle
    "swin_b"              # kaggle
]


for architecture in models_acr:
    cfg = get_training_config(architecture)
    print(f"[CONFIG] batch={cfg['batch_size']} | lr={cfg['lr']} | epochs={cfg['num_epochs']}")
    
    model, train_tfms, val_tfms = build_model_and_transforms(
    architecture,
    num_classes=num_classes,
    pretrained=True,
    strong_aug=True,
    device=device)

    model = freeze_backbone_unfreeze_head(model)

    train_dataset = SkinDataset(X_train_img, list(y_train), transform=train_tfms)
    val_dataset   = SkinDataset(X_val_img, list(y_val), transform=val_tfms)

    train_loader = DataLoader(train_dataset, batch_size=cfg["batch_size"], shuffle=True, num_workers=4)
    val_loader   = DataLoader(val_dataset, batch_size=cfg["batch_size"], shuffle=False, num_workers=4)

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg["lr"], weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=cfg["patience"]
    )

    model, train_losses, train_accs, val_losses, val_accs = train_model(
        model=model, train_loader=train_loader, val_loader=val_loader,
        num_epochs=cfg["num_epochs"], criterion=criterion, optimizer=optimizer,
        patience=cfg["patience"], unfreeze_fn=gradual_unfreeze,
        model_type=architecture, device=device, scheduler=scheduler,
        name_for_save=f"best_{architecture}"
    )

    duration = (time.time() - start_time) / 60
    best_val_acc = max(val_accs)
    results.append({
        "model": architecture,
        "best_val_acc": best_val_acc,
        "time_min": round(duration, 2)
    })
    print(f"✅ {architecture}: best val acc = {best_val_acc:.4f} ({duration:.1f} мин)")

In [ ]:
# Model prediction result
def evaluate_model(model, dataloader, device, class_names=None):
    model.eval()
    y_true, y_pred = [], []
    total_time, num_batches = 0.0, 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            if device.type == "cuda":
                torch.cuda.synchronize()
            start = time.time()
            outputs = model(images)
            if device.type == "cuda":
                torch.cuda.synchronize()
            end = time.time()

            total_time += (end - start)
            num_batches += 1

            _, predicted = torch.max(outputs.data, 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())

    # --- Метрики
    cm = confusion_matrix(y_true, y_pred)
    report = classification_report(y_true, y_pred, target_names=class_names, zero_division=0)
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted"),
        "avg_batch_time": total_time / num_batches,
        "avg_img_time": (total_time / num_batches) / dataloader.batch_size
    }

    print(f"\n🕒 Avg time per batch: {metrics['avg_batch_time']:.4f} sec")
    print(f"🖼️ Avg time per image: {metrics['avg_img_time']:.4f} sec")

    return metrics, cm, report

# Visualisation confusion matrix
def plot_confusion_matrix(cm, classes):
    with plt.style.context('default'):  
        plt.figure(figsize=(5, 4))
        sns.set(font_scale=1.0)
        sns.heatmap(cm, annot=True, fmt='g', cmap='Blues', cbar=False,
                    xticklabels=classes, yticklabels=classes)
        plt.xlabel('Predicted labels')
        plt.ylabel('True labels')
        plt.title('Confusion Matrix')
        plt.show()

In [ ]:
model_resnet18_base = torch.load("/kaggle/input/skin_resnet_18_base/pytorch/default/1/model_resnet18_base.pth",
                   map_location="cpu",
                   weights_only=False)
model_resnet18_base.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        _ = model_resnet18_base(images)
        break

metrics, cm, report = evaluate_model(model_resnet18_base, test_loader,device=device)
print("Metrics for current model:")
print(pd.DataFrame([metrics]))
print(report)
plot_confusion_matrix(cm, classes=list(range(num_classes)))